In [ ]:
import sys 
print(sys.executable)

/Users/yehjessica/Steven/bayesball-pitch-intent/venv/bin/python


In [ ]:
# python -m pip install -r requirements.txt
import pandas as pd 
import numpy as np

import os
from pathlib import Path
from dotenv import load_dotenv

import duckdb

import requests 
import json 
import pybaseball
from pybaseball import statcast 
pybaseball.cache.enable()

### STEPS
1. get data from pybaseball / MLB API 
2. Parquet on Google Drive 
3. DuckDB reads Parquet 
4. staging / processed / feature tables 

In [10]:
# Google Drive for parquet storage
# read root from .env file
load_dotenv()
DRIVE_ROOT = Path(os.environ["DRIVE_ROOT"])

RAW_DIR = DRIVE_ROOT / "data" / "raw" # / "statcast"
STAGING_DIR = DRIVE_ROOT / "data" / "staging"
PROCESSED_DIR = DRIVE_ROOT / "data" / "processed"

def make_output_dirs(year: int, dir=RAW_DIR):
    """
    Make output directories for a given year.
    """
    
    OUTPUT_DIR = dir/ f"year={year}" / f"statcast_{year}.parquet"
    OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
    return OUTPUT_DIR

In [ ]:
# get data
# monthly download per year

# statcast era: 2015 - present (2025 as the latest full season)
years  = range(2015,2026)
# season runs from March to October 
months = [
        ("03-01", "03-31"), # game_type: 'S' for spring training, 'R' for regular season
        ("04-01", "04-30"),
        ("05-01", "05-31"),
        ("06-01", "06-30"),
        ("07-01", "07-31"),
        ("08-01", "08-31"),
        ("09-01", "09-30"),
        ("10-01", "10-31"),
    ]

def fetch_statcast_data(start_date, end_date):
    try:
        data = statcast(start_date, end_date)
        return data
    except Exception as e:
        print(f"Error fetching data for {start_date} to {end_date}: {e}")
        return pd.DataFrame()  # Return an empty DataFrame on error
    

    


In [6]:
# p2025mar = fetch_statcast_data("2025-03-01", "2025-03-31")
print(p2025mar.shape) # (63808, 118)
print(p2025mar.head()) # reverse order (latest data first)
print(p2025mar.tail()) # 2025-03-15

This is a large query, it may take a moment to complete
Skipping offseason dates


  6%|▌         | 1/17 [00:03<00:53,  3.36s/it]/Users/stevey/cpsc330arm/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
 12%|█▏        | 2/17 [00:09<01:19,  5.28s/it]/Users/stevey/cpsc330arm/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/Users/stevey/cpsc330arm/lib/python3.11/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_d

(63808, 118)
     pitch_type  game_date  release_speed  release_pos_x  release_pos_z  \
1844         SI 2025-03-31           96.6           0.22           7.01   
1912         SL 2025-03-31           86.8           -0.5           6.78   
1987         SI 2025-03-31           97.8           0.18           7.03   
1998         SI 2025-03-31           97.8           0.18           6.95   
2067         SI 2025-03-31           97.7           0.05           7.02   

          player_name  batter  pitcher     events    description  ...  \
1844  Bautista, Félix  608324   642585  field_out  hit_into_play  ...   
1912  Bautista, Félix  608324   642585        NaN           ball  ...   
1987  Bautista, Félix  646240   642585       walk           ball  ...   
1998  Bautista, Félix  646240   642585        NaN           foul  ...   
2067  Bautista, Félix  646240   642585        NaN  called_strike  ...   

      batter_days_until_next_game  api_break_z_with_gravity  api_break_x_arm  \
1844             

In [9]:
p2025mar.groupby("game_date")['game_type'].value_counts()
p2025mar['game_type'].unique()

array(['R', 'S'], dtype=object)

In [ ]:
# pseudo 
function download_statcast_batches(years, months, raw_root, metadata_path,
                                   redo=False, continue_on_error=True):

    # 1. verify storage location
    if not path_exists(raw_root):
        raise Error("Raw storage path is not accessible")

    ensure_directory_exists(raw_root)
    ensure_directory_exists(parent_of(metadata_path))

    # 2. load metadata if it exists, else initialize empty metadata table
    if file_exists(metadata_path):
        metadata = load_metadata(metadata_path)
    else:
        metadata = empty_table_with_columns([
            source, year, month, start_date, end_date,
            file_path, status, row_count, game_count,
            downloaded_at, error_message
        ])

    failed_chunks = []
    skipped_chunks = []
    success_chunks = []

    # 3. iterate over year-month combinations
    for year in years:
        for month in months:

            start_date, end_date = get_month_date_range(year, month)

            output_path = build_output_path(raw_root, year, month)
            temp_output_path = build_temp_output_path(raw_root, year, month)

            try:
                # 4. check if chunk already exists
                already_exists = file_exists(output_path)
                metadata_success_exists = metadata_has_success_record(metadata, year, month)

                if already_exists and not redo:
                    log_skip_in_memory_or_metadata(...)
                    skipped_chunks.append((year, month))
                    continue

                # optional: remove old temp file if present
                if file_exists(temp_output_path):
                    delete_file(temp_output_path)

                # 5. query data
                df = statcast(start_dt=start_date, end_dt=end_date)

                # 6. optional filter
                if "game_type" in df.columns:
                    df = filter_rows(df, game_type == "R")

                # 7. compute summary stats
                row_count = number_of_rows(df)
                game_count = count_unique(df["game_pk"]) if "game_pk" in columns else null

                # 8. write to temp parquet first
                ensure_directory_exists(parent_of(output_path))
                write_parquet(df, temp_output_path)

                # 9. validate temp file exists
                if not file_exists(temp_output_path):
                    raise Error("Temp parquet write failed")

                # 10. replace/move temp file to final path
                rename_file(temp_output_path, output_path)

                # 11. update metadata as success
                metadata = upsert_metadata_record(
                    metadata,
                    year=year,
                    month=month,
                    start_date=start_date,
                    end_date=end_date,
                    file_path=output_path,
                    status="success",
                    row_count=row_count,
                    game_count=game_count,
                    downloaded_at=current_timestamp(),
                    error_message=None
                )

                save_metadata(metadata, metadata_path)
                success_chunks.append((year, month))

            except Exception as e:

                # 12. update metadata as failed
                metadata = upsert_metadata_record(
                    metadata,
                    year=year,
                    month=month,
                    start_date=start_date,
                    end_date=end_date,
                    file_path=output_path,
                    status="failed",
                    row_count=None,
                    game_count=None,
                    downloaded_at=current_timestamp(),
                    error_message=str(e)
                )

                save_metadata(metadata, metadata_path)
                failed_chunks.append((year, month, str(e)))

                print(f"FAILED at year={year}, month={month}: {e}")

                if not continue_on_error:
                    raise RuntimeError(f"Pipeline stopped at year={year}, month={month}") from e

    # 13. return summary
    return {
        "success": success_chunks,
        "skipped": skipped_chunks,
        "failed": failed_chunks
    }

In [ ]:
# DuckDB for local databasing 
con = duckdb.connect("local_db/bayesball.duckdb")


# TODO: insert path for read_parquet 
con.execute("""
    CREATE OR REPLACE VIEW raw_statcast AS 
    SELECT * 
    FROM read_parquet('.')
""")


In [ ]:
# 2025 pitch level data 
# data_p2025 = statcast(start_dt='2025-01-01', end_dt='2025-12-31')

In [ ]:
data_p2025.shape